In [2]:
!pip install mlxtend

import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Veriyi oku ve ilk 5 satırı gör
df = pd.read_csv('Groceries_dataset.csv')
print(f"Toplam satır sayısı: {len(df)}")
df.head()

Toplam satır sayısı: 38765


,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk


In [5]:
# 2. Aynı müşteri ve aynı tarihe ait ürünleri tek bir sepet(transaction) listesinde toplayalım
transactions = df.groupby(['Member_number', 'Date'])['itemDescription'].apply(lambda x: list(set(x))).tolist()

print(f"Toplam Sepet (transaction) sayısı: {len(transactions)}")
print(f"Örnek İlk 3 Sepet: \n{transactions[:3]}")

Toplam Sepet (transaction) sayısı: 14963
Örnek İlk 3 Sepet: 
[['semi-finished bread', 'whole milk', 'yogurt', 'sausage'], ['pastry', 'salty snack', 'whole milk'], ['canned beer', 'misc. beverages']]


In [6]:
# 3. TransactionEncoder ile sepetleri ikili matrise (True/False) dönüştürelim
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_transformed = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Matris Boyutu (Sepet x Ürün Sayısı): {df_transformed.shape}")
df_transformed.head()

Matris Boyutu (Sepet x Ürün Sayısı): (14963, 167)


,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,beverages,bottled beer,bottled water,brandy,brown bread,butter,butter milk,cake bar,candles,candy,canned beer,canned fish,canned fruit,canned vegetables,cat food,cereals,chewing gum,chicken,chocolate,chocolate marshmallow,citrus fruit,cleaner,cling film/bags,cocoa drinks,coffee,condensed milk,cooking chocolate,cookware,cream,cream cheese,...,salt,salty snack,sauces,sausage,seasonal products,semi-finished bread,shopping bags,skin care,sliced cheese,snack products,soap,soda,soft cheese,softener,soups,sparkling wine,specialty bar,specialty cheese,specialty chocolate,specialty fat,specialty vegetables,spices,spread cheese,sugar,sweet spreads,syrup,tea,tidbits,toilet cleaner,tropical fruit,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [7]:
# 4. En az %0.52 sıklıkla birlikte görülen ürün gruplarını buluyoruz
frequent_itemsets = apriori(df_transformed, min_support=0.0052, use_colnames=True)

# Kaç adet sık ürün kümesi bulduğunu yazdıralım
print(f"Bulunan Sık Ürün Kümesi Sayısı: {len(frequent_itemsets)}")
frequent_itemsets.sort_values(by='support', ascending=False).head(10)

Bulunan Sık Ürün Kümesi Sayısı: 123


,support,itemsets
86,0.157923,(whole milk)
52,0.122101,(other vegetables)
65,0.110005,(rolls/buns)
74,0.097106,(soda)
87,0.085879,(yogurt)
66,0.069572,(root vegetables)
80,0.067767,(tropical fruit)
6,0.060683,(bottled water)
69,0.060349,(sausage)
19,0.053131,(citrus fruit)


In [9]:
# 5. Güven (confidence) eşiği 0.12 olan kuralları çıkaralım
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.12)

# Kuralları confidence değerine göre büyükten küçüğe sıralayıp ilk 5 tanesini seçelim
top_5_rules = rules.sort_values(by='confidence', ascending=False).head(5)

# Çıktıyı temiz ve okunabilir bir tablo olarak formatlayalım
result_table = top_5_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
result_table['antecedents'] = result_table['antecedents'].apply(lambda x: ', '.join(list(x)))
result_table['consequents'] = result_table['consequents'].apply(lambda x: ', '.join(list(x)))
result_table.columns = ['Öncül Ürün (X)', 'Ardıl Ürün (Y)', 'Support (Destek)', 'Confidence (Güven)', 'Lift(Kaldıraç)']

print("--- CONFIDENCE DEĞERİ EN YÜKSEK İLK 5 BİRLİKTELİK KURALI ---")
result_table.reset_index(drop=True)

--- CONFIDENCE DEĞERİ EN YÜKSEK İLK 5 BİRLİKTELİK KURALI ---


,Öncül Ürün (X),Ardıl Ürün (Y),Support (Destek),Confidence (Güven),Lift(Kaldıraç)
0,bottled beer,whole milk,0.007151,0.157817,0.999330
1,sausage,whole milk,0.008955,0.148394,0.939663
2,newspapers,whole milk,0.005614,0.144330,0.913926
3,domestic eggs,whole milk,0.005280,0.142342,0.901341
4,frankfurter,whole milk,0.005280,0.139823,0.885388
